# Consultas directas sobre Parquet

Viajes de taxis amarillos y verdes de enero a agosto de 2026, leídos desde data/raw sin cargarlos a una tabla.

Las salidas de este notebook fueron regeneradas con los archivos reales de enero a agosto de 2026.


In [1]:
from pathlib import Path
import sys
import platform
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

raiz = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts/laboratorio").is_dir())
sys.path.insert(0, str(raiz / "scripts"))
from laboratorio.sesion_consultas import Analisis
from laboratorio.parametros import Configuracion
from laboratorio.soporte_cuadernos import configurar_notebook, ejecutar_script, mostrar_indicador
from laboratorio.graficos import COLOR, NOMBRE, COLOR_ANIO, MESES, miles

configurar_notebook()
config = Configuracion.del_entorno()
analisis = Analisis(config, anios=(2026,))
con = analisis.con
consulta = analisis.consulta


> Resultado de la ejecución con datos reales 2026.

## 3.1 Archivos disponibles

Cuántos archivos hay por tipo de taxi y qué meses cubren, según los nombres de los archivos en data/raw.

In [2]:
consulta("inspeccion/01_inventario_parquet.sql")

-- Archivos Parquet por tipo de taxi y rango de meses
SELECT
    regexp_extract(file, '(yellow|green)_tripdata', 1) AS tipo,
    count(*) AS archivos,
    min(regexp_extract(file, '(\d{4}-\d{2})\.parquet', 1)) AS primer_mes,
    max(regexp_extract(file, '(\d{4}-\d{2})\.parquet', 1)) AS ultimo_mes
FROM archivos
GROUP BY tipo
ORDER BY tipo

-- 0.01 s


,tipo,archivos,primer_mes,ultimo_mes
0,green,8,2026-01,2026-08
1,yellow,8,2026-01,2026-08


> Resultado de la ejecución con datos reales 2026.

Hay 8 archivos de cada tipo, de enero a agosto de 2026. Septiembre a diciembre todavía no están publicados, así que las comparaciones mensuales de 2026 llegan hasta agosto.

## 3.2 Registros disponibles

El pie de cada Parquet guarda su número de registros, así que el conteo no necesita leer filas.

In [3]:
consulta("inspeccion/02_conteos_metadatos.sql")

-- Registros por tipo de taxi según el pie de cada Parquet, sin leer las filas
WITH archivos AS (
    SELECT
        regexp_extract(file_name, '(yellow|green)_tripdata', 1) AS tipo,
        num_rows,
        num_row_groups
    FROM parquet_file_metadata(['/workspace/data/raw/yellow/2026/yellow_tripdata_2026-01.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-02.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-03.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-04.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-05.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-06.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-07.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-08.parquet', '/workspace/data/raw/green/2026/green_tripdata_2026-01.parquet', '/workspace/data/raw/green/2026/green_tripdata_2026-02.parquet', '/workspace/data/raw/green/2026/green_tripdata_2026-03.parquet', '/workspace/da

,tipo,archivos,registros,grupos_de_filas
0,green,8,337114,8
1,yellow,8,29703355,32
2,total,16,30040469,40


> Resultado de la ejecución con datos reales 2026.

La ejecución contó 30,040,469 registros: 29,703,355 amarillos y 337,114 verdes. Los verdes representan 1.12% del total; las comparaciones entre tipos deben usar porcentajes, promedios y medianas además de conteos.

## 3.3 Columnas presentes

Columnas y tipos de cada tipo de taxi, leyendo todos sus archivos.

In [4]:
consulta("inspeccion/03_comparacion_columnas.sql")

-- Columnas y tipos de datos de taxis amarillos y verdes, en el orden de los archivos
WITH yellow AS (
    SELECT row_number() OVER () AS orden, column_name, column_type
    FROM (DESCRIBE SELECT * FROM read_parquet(['/workspace/data/raw/yellow/2026/yellow_tripdata_2026-01.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-02.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-03.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-04.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-05.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-06.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-07.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-08.parquet'], union_by_name = true))
),
green AS (
    SELECT row_number() OVER () AS orden, column_name, column_type
    FROM (DESCRIBE SELECT * FROM read_parquet(['/workspace/data/raw/green/2026/green_tripdata_2026-01.parquet', '/workspace/data/raw/green/202

-- 0.11 s


,columna,tipo_yellow,tipo_green
0,VendorID,INTEGER,INTEGER
1,tpep_pickup_datetime,TIMESTAMP,NaN
2,lpep_pickup_datetime,NaN,TIMESTAMP
3,tpep_dropoff_datetime,TIMESTAMP,NaN
4,lpep_dropoff_datetime,NaN,TIMESTAMP
5,passenger_count,BIGINT,BIGINT
6,trip_distance,DOUBLE,DOUBLE
7,RatecodeID,BIGINT,BIGINT
8,store_and_fwd_flag,VARCHAR,VARCHAR
9,PULocationID,INTEGER,INTEGER


> Resultado de la ejecución con datos reales 2026.

La comparación expone 25 nombres de columna entre ambos esquemas. Las fechas cambian de prefijo (tpep en amarillos, lpep en verdes); Airport_fee aparece en amarillos y ehail_fee y trip_type en verdes. La vista de sql/normalizacion_viajes.sql renombra las fechas a pickup_datetime y dropoff_datetime y une los dos tipos por nombre de columna.

Falta revisar si todos los archivos de un mismo tipo traen las mismas columnas.

In [5]:
consulta("inspeccion/04_variaciones_esquema.sql")

-- Columnas que no están en todos los archivos de un tipo o que cambian de tipo físico
WITH esquema AS (
    SELECT
        regexp_extract(file_name, '(yellow|green)_tripdata', 1) AS tipo,
        regexp_extract(file_name, '(\d{4}-\d{2})\.parquet', 1) AS mes,
        name AS columna,
        type AS tipo_fisico
    FROM parquet_schema(['/workspace/data/raw/yellow/2026/yellow_tripdata_2026-01.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-02.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-03.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-04.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-05.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-06.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-07.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-08.parquet', '/workspace/data/raw/green/2026/green_tripdata_2026-01.parquet', '/workspace/data/raw/green/2026/green_tripdata_2026-02.parque

-- 0.16 s


,tipo,columna,archivos_con_columna,archivos_tipo,desde,hasta,tipos_fisicos
0,green,request_source,3,8,2026-06,2026-08,BYTE_ARRAY
1,yellow,request_source,3,8,2026-06,2026-08,BYTE_ARRAY


> Resultado de la ejecución con datos reales 2026.

request_source aparece en 3 de 8 archivos de cada tipo, desde junio hasta agosto de 2026. La siguiente consulta muestra cómo lee DuckDB esos archivos con y sin union_by_name.

In [6]:
consulta("inspeccion/05_conciliacion_esquemas.sql")

-- Columnas que DuckDB expone al leer todos los archivos amarillos con y sin union_by_name
SELECT
    'sin union_by_name' AS lectura,
    count(*) AS columnas,
    list_contains(list(column_name), 'request_source') AS incluye_request_source
FROM (DESCRIBE SELECT * FROM read_parquet(['/workspace/data/raw/yellow/2026/yellow_tripdata_2026-01.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-02.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-03.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-04.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-05.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-06.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-07.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-08.parquet']))
UNION ALL
SELECT
    'con union_by_name',
    count(*),
    list_contains(list(column_name), 'request_source')
FROM (DESCRIBE SELECT * FROM read_parquet(['/workspace/data/raw/yel

-- 0.23 s


,lectura,columnas,incluye_request_source
0,sin union_by_name,20,False
1,con union_by_name,21,True


> Resultado de la ejecución con datos reales 2026.

En esta ejecución, sin union_by_name DuckDB expuso 20 columnas y no incluyó request_source; con union_by_name expuso 21 y sí la incluyó. Por eso todas las lecturas del proyecto usan union_by_name: las columnas ausentes quedan nulas en los meses que no las traen.

## 3.4 Tipos de datos

Las columnas compartidas conservan tipos compatibles entre ambos tipos de taxi: fechas TIMESTAMP, identificadores INTEGER, códigos BIGINT, montos y distancia DOUBLE, y banderas o request_source de texto. La tabla generada por la consulta 03 documenta las diferencias exactas por columna.

RatecodeID, payment_type, VendorID y trip_type son códigos y no cantidades, así que en el ejercicio 4 se agrupan y se traducen con el diccionario de datos en lugar de promediarse.

## 3.5 Muestra de registros

Muestra aleatoria de cada tipo con semilla fija, para que se repita en cada ejecución.

In [7]:
consulta("inspeccion/06_muestra_taxi_amarillo.sql")

-- Muestra aleatoria reproducible de viajes amarillos
SELECT *
FROM read_parquet(['/workspace/data/raw/yellow/2026/yellow_tripdata_2026-01.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-02.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-03.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-04.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-05.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-06.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-07.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-08.parquet'], union_by_name = true)
USING SAMPLE bernoulli(0.00002%) REPEATABLE (42)



-- 26.04 s


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,request_source
0,2,2026-01-29 10:04:10,2026-01-29 10:09:30,1,0.95,1,N,107,233,1,7.20,0.0,0.5,2.39,0.0,1.0,14.34,2.5,0.0,0.75,NaN
1,2,2026-03-31 09:59:52,2026-03-31 10:08:15,1,1.14,1,N,141,263,1,9.30,0.0,0.5,2.66,0.0,1.0,15.96,2.5,0.0,0.00,NaN
2,2,2026-04-28 14:48:54,2026-04-28 14:54:22,2,0.35,1,N,236,75,1,6.50,0.0,0.5,2.10,0.0,1.0,12.60,2.5,0.0,0.00,NaN
3,2,2026-05-15 08:39:32,2026-05-15 08:51:18,1,1.75,1,N,170,113,1,12.80,0.0,0.5,3.51,0.0,1.0,21.06,2.5,0.0,0.75,NaN
4,7,2026-05-20 19:27:16,2026-05-20 19:27:16,1,1.10,1,N,163,141,2,11.40,0.0,0.5,0.00,0.0,1.0,18.65,2.5,0.0,0.75,NaN
5,2,2026-06-01 14:43:14,2026-06-01 15:13:12,1,2.52,1,N,162,246,2,25.40,0.0,0.5,0.00,0.0,1.0,30.15,2.5,0.0,0.75,NaN
6,2,2026-06-03 16:15:59,2026-06-03 16:20:48,3,0.88,1,N,238,151,2,7.20,2.5,0.5,0.00,0.0,1.0,11.20,0.0,0.0,0.00,NaN
7,2,2026-07-23 11:32:25,2026-07-23 11:46:53,1,0.87,1,N,161,186,1,12.80,0.0,0.5,3.00,0.0,1.0,20.55,2.5,0.0,0.75,NaN
8,2,2026-07-18 12:22:25,2026-07-18 12:34:57,<NA>,1.93,<NA>,NaN,79,231,0,18.69,0.0,0.5,2.34,0.0,1.0,25.78,NaN,NaN,0.75,A
9,2,2026-07-27 05:57:57,2026-07-27 06:07:27,<NA>,2.43,<NA>,NaN,236,230,0,15.65,0.0,0.5,2.00,0.0,1.0,22.40,NaN,NaN,0.75,A


In [8]:
consulta("inspeccion/07_muestra_taxi_verde.sql")

-- Muestra aleatoria reproducible de viajes verdes
SELECT *
FROM read_parquet(['/workspace/data/raw/green/2026/green_tripdata_2026-01.parquet', '/workspace/data/raw/green/2026/green_tripdata_2026-02.parquet', '/workspace/data/raw/green/2026/green_tripdata_2026-03.parquet', '/workspace/data/raw/green/2026/green_tripdata_2026-04.parquet', '/workspace/data/raw/green/2026/green_tripdata_2026-05.parquet', '/workspace/data/raw/green/2026/green_tripdata_2026-06.parquet', '/workspace/data/raw/green/2026/green_tripdata_2026-07.parquet', '/workspace/data/raw/green/2026/green_tripdata_2026-08.parquet'], union_by_name = true)
USING SAMPLE bernoulli(0.002%) REPEATABLE (42)



-- 1.36 s


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,request_source
0,2,2026-01-12 14:40:56,2026-01-12 14:41:36,N,1,82,82,2,0.07,3.00,0.0,0.5,0.0,0.00,NaN,1.0,4.50,2,1,0.00,0.0,NaN
1,6,2026-01-20 07:38:53,2026-01-20 08:26:40,NaN,<NA>,61,125,<NA>,6.51,3.00,0.0,0.5,0.0,0.00,NaN,0.3,35.08,<NA>,<NA>,NaN,0.0,NaN
2,2,2026-02-15 07:58:38,2026-02-15 08:07:44,N,1,74,166,1,2.10,12.10,0.0,0.5,2.0,0.00,NaN,1.0,15.60,1,1,0.00,0.0,NaN
3,2,2026-04-06 21:02:15,2026-04-06 21:14:09,N,1,166,238,1,1.66,12.80,1.0,0.5,0.0,0.00,NaN,1.0,18.05,2,1,2.75,0.0,NaN
4,2,2026-05-05 09:10:00,2026-05-05 09:16:00,NaN,<NA>,145,193,<NA>,1.42,14.33,0.0,0.5,1.0,0.00,NaN,1.0,16.83,<NA>,<NA>,NaN,0.0,NaN
5,2,2026-06-11 09:12:37,2026-06-11 09:20:04,N,1,134,197,5,1.80,10.00,0.0,0.5,0.0,0.00,NaN,1.0,11.50,2,1,0.00,0.0,NaN
6,6,2026-06-12 17:52:17,2026-06-12 18:45:00,NaN,<NA>,238,51,<NA>,14.25,3.00,0.0,0.5,0.0,7.46,NaN,0.3,42.41,<NA>,<NA>,NaN,0.0,A
7,2,2026-08-04 09:27:25,2026-08-04 09:32:16,N,1,55,55,1,0.82,7.20,0.0,0.5,0.0,0.00,NaN,1.0,8.70,2,1,0.00,0.0,NaN


> Resultado de la ejecución con datos reales 2026.

En la muestra ya aparecen tres problemas: registros sin passenger_count, RatecodeID ni congestion_surcharge cuando payment_type es 0 en amarillos o nulo en verdes, un viaje amarillo que empieza y termina en el mismo segundo y un viaje verde de 0.07 millas.

## 3.6 Calidad de datos

SUMMARIZE resume cada columna con su rango, sus cuartiles y su porcentaje de nulos.

In [9]:
consulta("inspeccion/08_perfil_taxi_amarillo.sql")

-- Tipo, rango, cuartiles y porcentaje de nulos de cada columna de los viajes amarillos
SUMMARIZE SELECT * FROM read_parquet(['/workspace/data/raw/yellow/2026/yellow_tripdata_2026-01.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-02.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-03.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-04.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-05.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-06.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-07.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-08.parquet'], union_by_name = true)



-- 45.15 s


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,7,4,1.885739809526567,0.7155277514022315,2,2,2,29703355,0.00
1,tpep_pickup_datetime,TIMESTAMP,2001-01-01 09:23:58,2026-08-31 23:59:59,16867928,2026-04-30 15:29:29.519432,NaN,2026-03-03 08:29:14.037002,2026-04-30 05:43:22.43944,2026-06-26 05:20:24.733688,29703355,0.00
2,tpep_dropoff_datetime,TIMESTAMP,2001-01-01 16:09:38,2026-09-01 20:16:00,16587125,2026-04-30 15:47:08.186216,NaN,2026-03-03 14:56:40.565337,2026-05-01 02:47:29.46952,2026-06-26 01:41:53.28175,29703355,0.00
3,passenger_count,BIGINT,0,9,11,1.2494318488564,0.6529195072996711,1,1,1,29703355,25.98
4,trip_distance,DOUBLE,0.0,328522.2,7217,5.552940149691246,550.6497893230985,1.0240113097767913,1.856764314968851,3.8105586461797007,29703355,0.00
5,RatecodeID,BIGINT,1,99,7,4.527471444398553,18.000926540066917,1,1,1,29703355,25.98
6,store_and_fwd_flag,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,29703355,25.98
7,PULocationID,INTEGER,1,265,290,161.57793013617484,66.74656703251708,117,161,233,29703355,0.00
8,DOLocationID,INTEGER,1,265,298,161.05139342003622,70.72548569211261,108,162,234,29703355,0.00
9,payment_type,BIGINT,0,5,6,0.8621735154160195,0.646332434596167,0,1,1,29703355,0.00


In [10]:
consulta("inspeccion/09_perfil_taxi_verde.sql")

-- Tipo, rango, cuartiles y porcentaje de nulos de cada columna de los viajes verdes
SUMMARIZE SELECT * FROM read_parquet(['/workspace/data/raw/green/2026/green_tripdata_2026-01.parquet', '/workspace/data/raw/green/2026/green_tripdata_2026-02.parquet', '/workspace/data/raw/green/2026/green_tripdata_2026-03.parquet', '/workspace/data/raw/green/2026/green_tripdata_2026-04.parquet', '/workspace/data/raw/green/2026/green_tripdata_2026-05.parquet', '/workspace/data/raw/green/2026/green_tripdata_2026-06.parquet', '/workspace/data/raw/green/2026/green_tripdata_2026-07.parquet', '/workspace/data/raw/green/2026/green_tripdata_2026-08.parquet'], union_by_name = true)



-- 0.81 s


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,6,3,2.328351833504393,1.2771882973738604,2,2,2,337114,0.00
1,lpep_pickup_datetime,TIMESTAMP,2008-12-31 17:35:31,2026-08-31 23:58:28,332702,2026-05-02 13:25:51.096427,NaN,2026-03-05 12:36:36.827049,2026-05-02 15:05:23.462558,2026-06-29 08:14:09.163618,337114,0.00
2,lpep_dropoff_datetime,TIMESTAMP,2008-12-31 23:16:26,2026-09-02 09:39:37,396286,2026-05-02 13:46:38.833759,NaN,2026-03-05 13:29:47.188799,2026-05-01 03:30:03.053417,2026-06-29 07:41:41.557785,337114,0.00
3,store_and_fwd_flag,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,337114,14.47
4,RatecodeID,BIGINT,1,99,7,1.2549672434183374,1.0015327424296085,1,1,1,337114,14.47
5,PULocationID,INTEGER,1,265,263,97.32761024460568,56.57830275002367,74,75,104,337114,0.00
6,DOLocationID,INTEGER,1,265,266,142.8768458147689,77.24476821098119,75,140,229,337114,0.00
7,passenger_count,BIGINT,0,9,11,1.3006461144694266,0.9481007387491747,1,1,1,337114,14.47
8,trip_distance,DOUBLE,0.0,179830.92,2472,13.349507644298379,880.4035093234567,1.254025445179726,2.0660806796415927,3.69026549897289,337114,0.00
9,fare_amount,DOUBLE,-500.0,1676.7,4808,17.014101995170655,17.95862763527065,8.602896855429488,13.17717599475841,19.67538779093272,337114,0.00


> Resultado de la ejecución con datos reales 2026.

El perfil y la consulta de anomalías detectaron fechas fuera del mes del archivo, montos negativos, distancias extremas y nulos concentrados en ciertos códigos de pago. La tabla completa de rangos y nulos queda en la salida de las consultas 08 y 09; no se trasladan cifras de ejecuciones anteriores a esta interpretación.

Que los porcentajes coincidan sugiere que los nulos caen en los mismos registros. La siguiente consulta los cruza con la forma de pago.

In [11]:
consulta("inspeccion/10_ausencias_por_pago.sql")

-- Relación entre la forma de pago y los nulos en pasajeros, código de tarifa y recargos
SELECT
    taxi,
    payment_type,
    count(*) AS viajes,
    round(100 * count(*) / sum(count(*)) OVER (PARTITION BY taxi), 2) AS pct_viajes,
    count(*) FILTER (WHERE passenger_count IS NULL) AS sin_passenger_count,
    count(*) FILTER (WHERE RatecodeID IS NULL) AS sin_ratecode,
    count(*) FILTER (WHERE congestion_surcharge IS NULL) AS sin_congestion
FROM viajes
GROUP BY taxi, payment_type
ORDER BY taxi, payment_type



-- 1.06 s


,taxi,payment_type,viajes,pct_viajes,sin_passenger_count,sin_ratecode,sin_congestion
0,green,1,219980,65.25,0,0,0
1,green,2,65921,19.55,0,0,0
2,green,3,1688,0.50,0,0,0
3,green,4,750,0.22,0,0,0
4,green,<NA>,48775,14.47,48775,48775,48775
5,yellow,0,7716688,25.98,7716688,7716688,7716688
6,yellow,1,18941008,63.77,0,0,0
7,yellow,2,2708031,9.12,0,0,0
8,yellow,3,98138,0.33,0,0,0
9,yellow,4,239488,0.81,0,0,0


> Resultado de la ejecución con datos reales 2026.

La consulta confirma que los nulos de passenger_count, RatecodeID y congestion_surcharge se concentran en payment_type 0 para yellow (7,716,688 viajes; 25.98%) y payment_type nulo para green (48,775 viajes; 14.47%). Se conservan en esta etapa y las decisiones de filtrado quedan para los ejercicios posteriores.

Para el resto de problemas se cuentan los registros que rompen reglas de fecha, duración, distancia, montos o zona. El mes de cada archivo sale de su nombre.

In [12]:
inconsistencias = consulta("inspeccion/11_anomalias_viajes.sql").set_index("taxi").T
porcentaje = (100 * inconsistencias / inconsistencias.loc["viajes"]).round(3)
pd.concat({"registros": inconsistencias, "porcentaje": porcentaje}, axis=1)

-- Registros que rompen reglas de fecha, duración, distancia, montos o zona
SELECT
    taxi,
    count(*) AS viajes,
    count(*) FILTER (WHERE date_trunc('month', pickup_datetime) <> periodo) AS inicio_fuera_del_mes,
    count(*) FILTER (WHERE dropoff_datetime <= pickup_datetime) AS duracion_cero_o_negativa,
    count(*) FILTER (WHERE dropoff_datetime - pickup_datetime > INTERVAL 3 HOUR) AS duracion_mayor_3h,
    count(*) FILTER (WHERE trip_distance = 0) AS distancia_cero,
    count(*) FILTER (WHERE trip_distance > 100) AS distancia_mayor_100,
    count(*) FILTER (WHERE dropoff_datetime > pickup_datetime
        AND trip_distance / nullif(epoch(dropoff_datetime - pickup_datetime) / 3600, 0) > 80) AS velocidad_mayor_80,
    count(*) FILTER (WHERE fare_amount < 0) AS tarifa_negativa,
    count(*) FILTER (WHERE total_amount <= 0) AS total_cero_o_negativo,
    count(*) FILTER (WHERE passenger_count = 0) AS pasajeros_cero,
    count(*) FILTER (WHERE PULocationID IN (264, 265) OR DOLocation

-- 7.27 s


registros         porcentaje         
taxi                        yellow   green     yellow    green
viajes                    29703355  337114    100.000  100.000
inicio_fuera_del_mes           146      98      0.000    0.029
duracion_cero_o_negativa    371683     234      1.251    0.069
duracion_mayor_3h            10127    1287      0.034    0.382
distancia_cero              952231   12212      3.206    3.623
distancia_mayor_100           1223      72      0.004    0.021
velocidad_mayor_80            7987    1158      0.027    0.344
tarifa_negativa             157364     999      0.530    0.296
total_cero_o_negativo       167093    1566      0.563    0.465
pasajeros_cero               91359    4527      0.308    1.343
zona_desconocida            201486    5886      0.678    1.746

> Resultado de la ejecución con datos reales 2026.

La ejecución detectó, entre otros casos, distancia cero en 952,231 yellow (3.206%) y 12,212 green (3.623%); duración cero o negativa en 371,683 yellow (1.251%) y 234 green (0.069%); y 146 yellow y 98 green con inicio fuera del mes del archivo. También encontró 201,486 yellow y 5,886 green con zona desconocida, además de tarifas o totales no válidos. Son problemas observados, no filas eliminadas en este ejercicio.

Por último, las filas repetidas en todas sus columnas.

In [13]:
consulta("inspeccion/12_filas_repetidas.sql")

-- Filas idénticas en todas las columnas
-- Primero agrupa por el hash de cada fila, que ocupa poca memoria, y después confirma con todas las columnas
WITH yellow AS NOT MATERIALIZED (
    SELECT f.*, hash(f) AS h
    FROM read_parquet(['/workspace/data/raw/yellow/2026/yellow_tripdata_2026-01.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-02.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-03.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-04.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-05.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-06.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-07.parquet', '/workspace/data/raw/yellow/2026/yellow_tripdata_2026-08.parquet'], union_by_name = true) AS f
),
green AS NOT MATERIALIZED (
    SELECT f.*, hash(f) AS h
    FROM read_parquet(['/workspace/data/raw/green/2026/green_tripdata_2026-01.parquet', '/workspace/data/raw/green/2026/green_tripdata_2026

-- 15.71 s


,taxi,grupos_repetidos,filas_sobrantes
0,yellow,7,7
1,green,0,0


> Resultado de la ejecución con datos reales 2026.

Se encontraron 7 grupos repetidos y 7 filas sobrantes en yellow; green no presentó grupos ni filas sobrantes repetidas.

Con estos resultados, sql/normalizacion_viajes.sql define la vista viajes_validos. Conserva los viajes que empiezan en el mes de su archivo, terminan después de empezar, duran hasta 3 horas, recorren más de 0 y hasta 100 millas, no pasan de 80 millas por hora, no tienen tarifa negativa y tienen total mayor que cero. Las siete filas repetidas se dejan porque no cambian ningún resultado, y las zonas 264 y 265 quedan como desconocidas.

In [14]:
analisis.crear_vistas()
consulta("inspeccion/13_resumen_calidad.sql")

-- Registros que conserva la vista viajes_validos (requiere sql/normalizacion_viajes.sql)
WITH todos AS (
    SELECT taxi, count(*) AS registros FROM viajes GROUP BY taxi
),
validos AS (
    SELECT taxi, count(*) AS validos FROM viajes_validos GROUP BY taxi
)
SELECT
    taxi,
    registros,
    validos,
    registros - validos AS excluidos,
    round(100 * validos / registros, 2) AS pct_validos
FROM todos
JOIN validos USING (taxi)
ORDER BY taxi DESC



-- 3.62 s


,taxi,registros,validos,excluidos,pct_validos
0,yellow,29703355,28223766,1479589,95.02
1,green,337114,321395,15719,95.34


> Resultado de la ejecución con datos reales 2026.

La vista de calidad conserva 28,223,766 de 29,703,355 yellow (95.02%) y 321,395 de 337,114 green (95.34%). La consulta registra explícitamente los excluidos para que la decisión sea reproducible.

## 3.7 y 3.8 Consultas sobre los archivos

Las trece consultas están en sql/inspeccion y leen los Parquet con parquet_file_metadata, parquet_schema, read_parquet o vistas sobre read_parquet. No se creó ninguna tabla materializada y la conexión de DuckDB vive en memoria.

## 3.9 Consultar un Parquet sin importarlo

DuckDB lee el archivo en el momento de la consulta, sin cargarlo antes a una tabla. Un Parquet guarda los datos por columnas, en grupos de filas, con un pie que tiene el esquema, el número de registros y el mínimo y máximo de cada columna en cada grupo. Con eso DuckDB lee las columnas que usa la consulta y omite las demás, salta los grupos de filas que no cumplen el filtro, responde conteos y esquemas desde el pie, como en 3.2, y reparte el trabajo entre los núcleos del procesador.

Con volúmenes grandes eso evita copiar los datos a otro formato y esperar una carga, no exige que el conjunto quepa en memoria y deja cualquier archivo nuevo de data/raw disponible en la siguiente consulta. El costo de cada consulta depende de las columnas y grupos que lee y no del tamaño total de los archivos: SUMMARIZE recorrió las 21 columnas de 29.7 millones de viajes en segundos.

In [15]:
analisis.cerrar()
